# TalkNet-ASD: End-to-End Inference Demo
Runs TalkNet-ASD Active Speaker Detection on Google Colab GPU.

**Setup:** `Runtime -> Change runtime type -> T4 GPU`, then **Run All**.

In [ ]:
# 1. Setup Environment
import os
os.chdir("/content")

!rm -rf TalkNet-ASD
!git clone https://github.com/TaoRuijie/TalkNet-ASD.git
os.chdir("/content/TalkNet-ASD")
print(f"Working dir: {os.getcwd()}")

# Install dependencies
!pip install -q scenedetect==0.5.6.1
!pip install -q gdown scipy librosa opencv-python python_speech_features tqdm yt-dlp
!apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null 2>&1

# Patch deprecated np.int/np.float/np.bool for NumPy 1.24+
!find . -name "*.py" -exec sed -i "s/np\.int)/int)/g" {} +
!find . -name "*.py" -exec sed -i "s/np\.int,/int,/g" {} +
!find . -name "*.py" -exec sed -i "s/np\.int]/int]/g" {} +
!find . -name "*.py" -exec sed -i "s/np\.float)/float)/g" {} +
!find . -name "*.py" -exec sed -i "s/np\.bool)/bool)/g" {} +

print("\n=== Setup complete ===")

In [ ]:
# 2. Download a real podcast/interview clip
import os
os.chdir("/content/TalkNet-ASD")

!mkdir -p demo

# Download a 90-second segment from a Lex Fridman podcast (two people, face-to-face, taking turns talking)
!yt-dlp -f "bv*[ext=mp4]+ba[ext=m4a]/b[ext=mp4]" \
    --download-sections "*00:00:30-00:02:00" \
    --force-keyframes-at-cuts \
    -o "demo/podcast_clip.%(ext)s" \
    "https://www.youtube.com/watch?v=DxREm3s1scA"

!ls -lh demo/podcast_clip.mp4
print("\n=== Video ready ===")

In [ ]:
# 3. Run TalkNet-ASD Inference
import os
os.chdir("/content/TalkNet-ASD")
print(f"Working dir: {os.getcwd()}")
print(f"demoTalkNet.py exists: {os.path.exists(demoTalkNet.py)}")
print(f"Video exists: {os.path.exists(demo/podcast_clip.mp4)}")

# Run inference (pretrained checkpoint auto-downloads on first run)
!python demoTalkNet.py --videoName podcast_clip

print("\n=== Inference complete ===")

In [ ]:
# 4. Check Results
import os
os.chdir("/content/TalkNet-ASD")

output_path = "demo/podcast_clip/pyavi/video_out.avi"

if os.path.exists(output_path):
    size = os.path.getsize(output_path) / (1024*1024)
    print(f"\n=== SUCCESS! Output: {output_path} ({size:.1f} MB) ===")
else:
    print("Output not at expected path. Searching...")
    for root, dirs, files in os.walk("demo/"):
        for f in files:
            full = os.path.join(root, f)
            size = os.path.getsize(full) / (1024*1024)
            print(f"  {full} ({size:.1f} MB)")

In [ ]:
# 5. Download result to your Mac
import os
os.chdir("/content/TalkNet-ASD")
from google.colab import files

output_path = "demo/podcast_clip/pyavi/video_out.avi"
if os.path.exists(output_path):
    files.download(output_path)
else:
    # Try to find and download whatever output exists
    for root, dirs, flist in os.walk("demo/podcast_clip/"):
        for f in flist:
            if f.endswith(".avi"):
                files.download(os.path.join(root, f))